In [16]:
import numpy as np

print("NumPy version:", np.__version__)

NumPy version: 2.5.4


In [17]:
machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

In [18]:
import csv

# County names
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

# Rainfall data for the first four counties
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]
]

print("Counties:", counties)
print("Number of rainfall records:", len(clean_rows))

Counties: ['Nairobi', 'Nakuru', 'Kisumu', 'Mombasa', 'Machakos']
Number of rainfall records: 4


In [19]:
def clean_machakos_data(raw_data, month_names):

    cleaned_values = []
    valid_values = []
    imputed_months = []

    for month, value in zip(month_names, raw_data):

        try:
            rainfall = int(value.strip())
            cleaned_values.append(rainfall)
            valid_values.append(rainfall)

        except ValueError:
            cleaned_values.append(None)
            imputed_months.append(month)

    # Calculate rounded average of valid readings
    average_rainfall = round(np.mean(valid_values))

    # Replace invalid values
    cleaned_values = [
        average_rainfall if value is None else value
        for value in cleaned_values
    ]

    print("Average rainfall used for imputation:", average_rainfall, "mm")
    print("\nImputed months:")

    for month in imputed_months:
        print(f"{month}: {average_rainfall} mm")

    return cleaned_values


# Apply cleaning function
machakos_clean = clean_machakos_data(machakos_raw, months)

print("\nCleaned Machakos data:")
print(machakos_clean)

Average rainfall used for imputation: 68 mm

Imputed months:
April: 68 mm
September: 68 mm

Cleaned Machakos data:
[48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [20]:
# Combine the four clean county datasets with Machakos
all_rows = clean_rows + [machakos_clean]

# Convert the combined data into a NumPy array
R = np.array(all_rows)

# Display the rainfall array
print("COMBINED RAINFALL DATA:")
print(R)

# Check array properties
print("\nArray shape:", R.shape)
print("Array data type:", R.dtype)


COMBINED RAINFALL DATA:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]

Array shape: (5, 12)
Array data type: int64


In [21]:
# Calculate annual rainfall totals for each county
annual_totals = np.sum(R, axis=1)

# Calculate average monthly rainfall across all counties
monthly_averages = np.mean(R, axis=0)

# Display annual totals
print("ANNUAL RAINFALL TOTALS")
print("=" * 40)

for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")


# Display monthly averages
print("\nAVERAGE MONTHLY RAINFALL")
print("=" * 40)

for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.2f} mm")
    

ANNUAL RAINFALL TOTALS
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

AVERAGE MONTHLY RAINFALL
January: 47.00 mm
February: 47.60 mm
March: 91.00 mm
April: 164.40 mm
May: 171.20 mm
June: 67.60 mm
July: 56.60 mm
August: 58.20 mm
September: 64.20 mm
October: 67.80 mm
November: 127.40 mm
December: 80.60 mm


In [22]:
# Find the index of the wettest month for each county
wettest_indices = np.argmax(R, axis=1)

# Convert the indices into month names
wettest_months = np.array(months)[wettest_indices]

# Display the results
print("WETTEST MONTH PER COUNTY")
print("=" * 40)

for county, month in zip(counties, wettest_months):
    print(f"{county}: {month}")


WETTEST MONTH PER COUNTY
Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


In [23]:
# Create a Boolean mask for rainfall below 50 mm
dry_mask = R < 50

# Count dry months for each county
dry_month_counts = np.sum(dry_mask, axis=1)

# Display results
print("DRY MONTH COUNTS")
print("=" * 40)

for county, count in zip(counties, dry_month_counts):
    print(f"{county}: {count} dry months")

DRY MONTH COUNTS
Nairobi: 4 dry months
Nakuru: 3 dry months
Kisumu: 0 dry months
Mombasa: 2 dry months
Machakos: 6 dry months


In [24]:
# Calculate combined rainfall for consecutive months
consecutive_rainfall = R[:, :-1] + R[:, 1:]

# Find the index of the highest combined rainfall for each county
planting_indices = np.argmax(consecutive_rainfall, axis=1)

# Identify the corresponding two-month planting windows
planting_windows = [
    f"{months[index]} - {months[index + 1]}"
    for index in planting_indices
]

# Display results
print("RECOMMENDED PLANTING WINDOWS")
print("=" * 45)

for county, window in zip(counties, planting_windows):
    print(f"{county}: {window}")

RECOMMENDED PLANTING WINDOWS
Nairobi: April - May
Nakuru: April - May
Kisumu: April - May
Mombasa: April - May
Machakos: November - December


In [25]:
# Define CSV column headers
headers = [
    "county",
    "annual_total_mm",
    "wettest_month",
    "dry_months",
    "planting_window"
]

# Create the CSV file
with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as file:

    writer = csv.writer(file)

    # Write headers
    writer.writerow(headers)

    # Write five county records
    writer.writerows(
        zip(
            counties,
            annual_totals,
            wettest_months,
            dry_month_counts,
            planting_windows
        )
    )

print("rainfall_summary.csv created successfully!")

rainfall_summary.csv created successfully!


In [26]:
# Read and verify the CSV file
with open("rainfall_summary.csv", "r", encoding="utf-8") as file:

    reader = csv.reader(file)

    for row in reader:
        print(row)

['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
['Nairobi', '1045', 'April', '4', 'April - May']
['Nakuru', '916', 'April', '3', 'April - May']
['Kisumu', '1371', 'April', '0', 'April - May']
['Mombasa', '1075', 'May', '2', 'April - May']
['Machakos', '811', 'November', '6', 'November - December']


## Planting Advisory

The analysis identifies April–May as the recommended planting window for Nairobi, Nakuru, Kisumu and Mombasa because these months have the highest combined consecutive rainfall. Machakos has a different rainfall pattern, with November–December recording its highest combined rainfall. Kisumu records the highest annual rainfall at 1,371 mm, while Machakos records the lowest at 811 mm. Farmers should use these historical rainfall patterns to guide planting preparation while considering soil conditions, crop requirements and current weather forecasts. These recommendations are based on historical monthly readings and should not be treated as guaranteed predictions of future rainfall.

## Data Quality Notes

The original Machakos dataset contained two invalid entries: "N/A" for April and an empty string for September. Both entries were handled using a specific ValueError exception during integer conversion. The invalid readings were replaced with the rounded average of the ten valid observations, resulting in an imputed value of 68 mm for each month. This approach assumes that the available readings provide a reasonable estimate of the missing observations. Consequently, Machakos rainfall totals and planting recommendations should be interpreted with caution because two readings were estimated rather than directly measured.